In [1]:
!pip install playwright nest-asyncio
!playwright install chromium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 MB 21.8 MB/s eta 0:00:00
170.4 MiB [] 0% 0.0s170.4 MiB [] 0% 218.9s170.4 MiB [] 0% 240.3s170.4 MiB [] 0% 825.8s170.4 MiB [] 0% 681.9s170.4 MiB [] 0% 938.1s170.4 MiB [] 0% 819.7s170.4 MiB [] 0% 738.3s170.4 MiB [] 0% 677.2s170.4 MiB [] 0% 629.7s170.4 MiB [] 0% 647.3s170.4 MiB [] 0% 606.2s170.4 MiB [] 0% 576.5s170.4 MiB [] 0% 551.4s170.4 MiB [] 0% 529.1s170.4 MiB [] 0% 509.7s170.4 MiB [] 0% 491.4s170.4 MiB [] 0% 474.0s170.4 MiB [] 0% 458.5s170.4 MiB [] 0% 447.5s170.4 MiB [] 0% 416.7s170.4 MiB [] 0% 391.3s170.4 MiB [] 0% 369.9s170.4 MiB [] 0% 351.7s170.4 MiB [] 0% 336.8s170.4 MiB [] 0% 331.7s170.4 MiB [] 0% 318.4s170.4 MiB [] 0% 316.6s170.4 MiB [] 0% 296.4s170.4 MiB [] 0% 285.1s170.4 MiB [] 0% 270.2s170.4 MiB [] 0% 256.4s170.4 MiB [] 0% 243.9s170.4 MiB [] 0% 232.7s170.4 MiB [] 0% 222.7s170.4 MiB [] 0% 214.6s170.4 MiB [] 0% 203.7s170.4 MiB [] 0% 203.9s170.4 MiB [] 0% 198.5s170.4 MiB [] 0% 187.3s170.4 MiB [] 0% 179.3s170.4 MiB [] 0% 170.6

In [2]:
!pip install playwright faker nest-asyncio > /dev/null
!playwright install chromium
!playwright install-deps > /dev/null
print("✅ Dependencies installed")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Failed to fetch https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu/dists/jammy/InRelease  503  Service Unavailable [IP: 185.125.190.80 443]
W: Failed to fetch https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu/dists/jammy/InRelease  Could not connect to ppa.launchpadcontent.net:443 (185.125.190.80), connection timed out [IP: 185.125.190.80 443]
W: Failed to fetch https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu/dists/jammy/InRelease  Unable to connect to ppa.launchpadcontent.net:443: [IP: 185.125.190.80 443]
W: Some index files failed to download. They have been ignored, or old ones used instead.
Extracting templates from packages: 100%
✅ Dependencies installed


In [ ]:
import asyncio
import sys
import os
import random
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError
import nest_asyncio
nest_asyncio.apply()

# Indian names list instead of faker
INDIAN_NAMES = [
    "Aarav Sharma", "Vihaan Gupta", "Vivaan Kumar", "Ananya Singh", "Diya Verma",
    "Advik Reddy", "Kabir Joshi", "Aaradhya Nair", "Sai Patel", "Ishaan Malhotra",
    "Sanya Kapoor", "Rohan Mehta", "Priya Krishnamurthy", "Arjun Rao", "Neha Desai",
    "Karthik Iyer", "Anjali Shetty", "Rahul Choudhary", "Pooja Mishra", "Vikram Singh",
    "Aditi Sharma", "Rajesh Khanna", "Deepika Padukone", "Amitabh Bachchan", "Priyanka Chopra",
    "Ranbir Kapoor", "Alia Bhatt", "Shah Rukh Khan", "Aishwarya Rai", "Salman Khan",
    "Rajnikanth", "Kamal Haasan", "Nayanthara", "Vijay Sethupathi", "Madhuri Dixit",
    "Akshay Kumar", "Kareena Kapoor", "Hrithik Roshan", "Katrina Kaif", "Ranveer Singh"
]

print("\n===== ZOOM AUTO JOIN BOT =====\n")
MEETING_CODE = input("Enter Meeting ID (no spaces): ").strip().replace(" ", "")
PASSCODE     = input("Enter Passcode: ").strip()
USERS        = int(input("How many users to join?: ").strip())
WAIT_MIN     = int(input("Stay for how many minutes?: ").strip())

# Fully headless — NO visible browser windows. One Chromium process for all users.
# Honors env var ZOOM_BOT_HEADLESS=0 to disable for debugging; otherwise always True.
HEADLESS     = os.environ.get("ZOOM_BOT_HEADLESS", "1") != "0"
DEBUG_DIR    = "zoom_bot_debug"
os.makedirs(DEBUG_DIR, exist_ok=True)

print(f"\nPYTHON: {sys.version}")
print(f"Meeting: {MEETING_CODE} | Users: {USERS} | Duration: {WAIT_MIN} min")
if not HEADLESS:
    print("⚠️  HEADLESS=False — browser windows WILL open. "
          "Unset ZOOM_BOT_HEADLESS or set it to 1 to hide them.")
print(f"Headless: {HEADLESS} (no windows will open if True) | Debug: ./{DEBUG_DIR}/\n")


# Stealth init script — Zoom's web client rejects browsers it identifies as automated.
# This patches the most common detection signals so headless Chromium looks like a real Chrome.
STEALTH_JS = r"""
Object.defineProperty(navigator, 'webdriver', { get: () => undefined });
Object.defineProperty(navigator, 'languages', { get: () => ['en-US', 'en'] });
Object.defineProperty(navigator, 'plugins', {
    get: () => [
        { name: 'PDF Viewer', filename: 'internal-pdf-viewer', description: '' },
        { name: 'Chrome PDF Viewer', filename: 'internal-pdf-viewer', description: '' },
        { name: 'Chromium PDF Viewer', filename: 'internal-pdf-viewer', description: '' },
    ],
});
Object.defineProperty(navigator, 'mimeTypes', {
    get: () => [{ type: 'application/pdf', suffixes: 'pdf', description: '' }],
});
window.chrome = window.chrome || { runtime: {}, app: {}, csi: () => {}, loadTimes: () => {} };
const origQuery = navigator.permissions && navigator.permissions.query;
if (origQuery) {
    navigator.permissions.query = (params) =>
        params && params.name === 'notifications'
            ? Promise.resolve({ state: Notification.permission })
            : origQuery.call(navigator.permissions, params);
}
Object.defineProperty(navigator, 'hardwareConcurrency', { get: () => 8 });
Object.defineProperty(navigator, 'deviceMemory', { get: () => 8 });
"""

REAL_UA = ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
           "AppleWebKit/537.36 (KHTML, like Gecko) "
           "Chrome/124.0.0.0 Safari/537.36")


async def find_first(page, selectors, timeout=8000):
    for s in selectors:
        try:
            loc = page.locator(s)
            await loc.first.wait_for(state="visible", timeout=timeout)
            return loc.first
        except Exception:
            continue
    return None


async def snap(page, tag, label):
    """Save a screenshot + url + visible text so we can diagnose where the bot got stuck."""
    safe = tag.strip("[]").replace(" ", "_")
    path = os.path.join(DEBUG_DIR, f"{safe}_{label}.png")
    try:
        await page.screenshot(path=path, full_page=True)
        url = page.url
        try:
            text = await page.evaluate("() => document.body ? document.body.innerText.slice(0, 400) : ''")
        except Exception:
            text = ""
        print(f"{tag} 📸 saved {path}")
        print(f"{tag}    URL : {url}")
        if text.strip():
            print(f"{tag}    TEXT: {text.strip()[:300].replace(chr(10), ' | ')}")
    except Exception as e:
        print(f"{tag} ⚠️  could not screenshot: {e}")


async def click_join_from_browser_if_present(page, tag):
    """Zoom often shows an 'Open Zoom Workplace?' page with a small 'Join from your browser' link."""
    for sel in [
        'a:has-text("Join from your browser")',
        'a:has-text("Join from Your Browser")',
        'a.web_client_link',
        '#webclient',
    ]:
        try:
            loc = page.locator(sel).first
            if await loc.is_visible(timeout=1500):
                await loc.click(force=True)
                print(f"{tag} 🌐 clicked 'Join from your browser'")
                await page.wait_for_timeout(2500)
                return True
        except Exception:
            continue
    return False


async def accept_terms_if_present(page, tag):
    for sel in [
        'button:has-text("I Agree")',
        'button:has-text("Accept")',
        'button:has-text("Agree")',
        '#wc_agree',
    ]:
        try:
            loc = page.locator(sel).first
            if await loc.is_visible(timeout=1000):
                await loc.click(force=True)
                print(f"{tag} 📜 accepted terms")
                await page.wait_for_timeout(1500)
                return True
        except Exception:
            continue
    return False


async def verify_in_meeting(page, tag, timeout=30000):
    """Return True only when the in-meeting toolbar is actually visible."""
    in_meeting_selectors = [
        "button[aria-label='Leave']",
        "button[aria-label*='Leave']",
        ".footer__leave-btn",
        "button.zm-btn--leave",
        "button[aria-label='Participants']",
        "button[aria-label*='Participants']",
        "#foot-bar",
        ".meeting-app",
    ]
    elapsed = 0
    step = 1000
    while elapsed < timeout:
        try:
            wr = page.locator('text=/please wait.*host.*let you in/i').first
            if await wr.is_visible(timeout=300):
                print(f"{tag} 🚪 stuck in WAITING ROOM — host must Admit this bot.")
                return False
        except Exception:
            pass
        for err_sel in [
            'text=/host has.*not.*started/i',
            'text=/meeting has been locked/i',
            'text=/you have been removed/i',
            'text=/connection.*timed out/i',
            'text=/your meeting has been ended/i',
        ]:
            try:
                if await page.locator(err_sel).first.is_visible(timeout=300):
                    txt = await page.locator(err_sel).first.text_content()
                    print(f"{tag} ❌ blocked by Zoom: {txt!r}")
                    return False
            except Exception:
                pass
        for s in in_meeting_selectors:
            try:
                if await page.locator(s).first.is_visible(timeout=300):
                    return True
            except Exception:
                continue
        await page.wait_for_timeout(step)
        elapsed += step
    return False


async def join_meeting(browser, tag, meeting_code, passcode, wait_seconds):
    """Each user gets its own isolated browser CONTEXT (cookies, storage) but shares
    a single Chromium process — so only one background process for all users."""
    print(f"{tag} 🚀 Starting ...")

    context = await browser.new_context(
        permissions=["camera", "microphone"],
        viewport={"width": 1280, "height": 720},
        user_agent=REAL_UA,
        locale="en-US",
    )
    await context.add_init_script(STEALTH_JS)
    page = await context.new_page()

    try:
        url = f"https://app.zoom.us/wc/{meeting_code}/join?fromPWA=1&pwd={passcode}"
        print(f"{tag} 🌐 Navigating ...")
        try:
            await page.goto(url, timeout=120000, wait_until="domcontentloaded")
        except Exception as e:
            print(f"{tag} ❌ navigation failed: {e}")
            await snap(page, tag, "nav_fail")
            return
        await page.wait_for_timeout(3500)

        try:
            await page.evaluate("""
                () => document.querySelectorAll('[id*="onetrust"],[class*="onetrust"]')
                              .forEach(e => e.remove())
            """)
        except Exception:
            pass

        await click_join_from_browser_if_present(page, tag)
        await accept_terms_if_present(page, tag)
        await page.wait_for_timeout(1500)

        # ── Passcode ──
        pass_input = await find_first(page, [
            "#input-for-pwd",
            'input[aria-describedby="error-for-pwd"]',
            'input[placeholder*="passcode" i]',
            'input[type="password"]',
        ], timeout=4000)
        if pass_input:
            await pass_input.click(force=True)
            await pass_input.fill(passcode)
            print(f"{tag} 🔑 Passcode filled.")
        else:
            print(f"{tag} ℹ️  no passcode field (already in URL).")

        await page.wait_for_timeout(500)

        # ── Name (handles new "Enter Meeting Info" UI where the input has no placeholder) ──
        name = random.choice(INDIAN_NAMES)
        name_input = None

        # 1) Label-based — works on the new UI where the only hint is the "Your Name" label.
        try:
            loc = page.get_by_label("Your Name", exact=False).first
            await loc.wait_for(state="visible", timeout=5000)
            name_input = loc
        except Exception:
            pass

        # 2) Fall back to ID / name / placeholder / any visible non-password text input.
        if not name_input:
            name_input = await find_first(page, [
                "#input-for-name",
                "#inputname",
                'input[name="inputname"]',
                'input[placeholder*="name" i]',
                'input[aria-label*="name" i]',
                'input[type="text"]',
                'input:not([type="password"]):not([type="hidden"]):not([type="checkbox"])',
            ], timeout=8000)

        if name_input:
            await name_input.click(force=True)
            try:
                await name_input.fill("")
            except Exception:
                pass
            # Type char-by-char so React/Vue input listeners fire and the Join button enables.
            await name_input.type(name, delay=40)

            # Verify the value actually stuck — if not, set it via the native value setter
            # and dispatch input/change events (React requires this exact dance).
            try:
                actual = await name_input.input_value()
            except Exception:
                actual = ""
            if (actual or "").strip() != name:
                await page.evaluate(
                    """(val) => {
                        const candidates = Array.from(document.querySelectorAll('input'))
                            .filter(el => el.offsetParent !== null
                                       && el.type !== 'password'
                                       && el.type !== 'hidden'
                                       && el.type !== 'checkbox');
                        const el = candidates.find(e => /name/i.test(e.getAttribute('aria-label') || ''))
                                || candidates.find(e => /name/i.test(e.placeholder || ''))
                                || candidates[0];
                        if (!el) return false;
                        const setter = Object.getOwnPropertyDescriptor(window.HTMLInputElement.prototype, 'value').set;
                        setter.call(el, val);
                        el.dispatchEvent(new Event('input',  {bubbles: true}));
                        el.dispatchEvent(new Event('change', {bubbles: true}));
                        return true;
                    }""",
                    name,
                )
            print(f"{tag} 📝 Name set to '{name}'")
        else:
            print(f"{tag} ⚠️  Name field not found — taking screenshot.")
            await snap(page, tag, "no_name_field")

        await page.wait_for_timeout(500)

        # ── Pre-join: turn off camera ──
        try:
            cam_btn = await find_first(page, [
                "button[aria-label='Stop Video']",
                "button[aria-label='Turn off video']",
                "button.preview-video-control",
            ], timeout=2500)
            if cam_btn:
                await cam_btn.click(force=True)
                print(f"{tag} 📷 Camera off (pre-join).")
        except Exception:
            pass

        # ── Join ──
        join_btn = await find_first(page, [
            'button.preview-join-button',
            'button:has-text("Join")',
            'button:has-text("Join Meeting")',
            'button[type="submit"]',
        ], timeout=8000)
        if join_btn:
            await join_btn.click(force=True)
            print(f"{tag} ✅ Join clicked — verifying admission ...")
        else:
            print(f"{tag} ❌ Join button not found.")
            await snap(page, tag, "no_join_btn")
            return

        # Audio dialog
        await page.wait_for_timeout(3000)
        for sel in [
            'button:has-text("Join Audio by Computer")',
            'button:has-text("Computer Audio")',
            'button[aria-label*="Join Audio"]',
        ]:
            try:
                loc = page.locator(sel).first
                if await loc.is_visible(timeout=1500):
                    await loc.click(force=True)
                    print(f"{tag} 🔊 joined computer audio")
                    break
            except Exception:
                continue

        # ── VERIFY ──
        if not await verify_in_meeting(page, tag, timeout=30000):
            print(f"{tag} ❌ NEVER ENTERED MEETING — saving evidence.")
            await snap(page, tag, "not_in_meeting")
            return

        print(f"{tag} 🟢 IN MEETING as '{name}'.")

        # In-meeting: camera off
        try:
            await page.evaluate("""
                () => {
                    for (const btn of document.querySelectorAll('button')) {
                        const label = (btn.getAttribute('aria-label') || btn.innerText || '').toLowerCase();
                        if (label.includes('stop video') && !btn.disabled) { btn.click(); return true; }
                    }
                    return false;
                }
            """)
        except Exception:
            pass

        print(f"{tag} ⏳ staying for {wait_seconds // 60} min ...")
        for remaining in range(wait_seconds, 0, -30):
            mins, secs = divmod(remaining, 60)
            print(f"{tag} ⏱️  {mins:02d}:{secs:02d} remaining", end="\r")
            await asyncio.sleep(30)

        # Leave
        print(f"\n{tag} 👋 leaving.")
        try:
            leave = await find_first(page, [
                ".zm-btn--leave",
                "button[aria-label='Leave']",
                "button:has-text('Leave')",
            ], timeout=5000)
            if leave:
                await leave.click(force=True)
                await page.wait_for_timeout(1500)
                confirm = await find_first(page, [
                    "button:has-text('Leave Meeting')",
                ], timeout=4000)
                if confirm:
                    await confirm.click(force=True)
                    print(f"{tag} ✅ Left.")
        except Exception:
            pass

    finally:
        await context.close()
        print(f"{tag} ✅ Done.")


async def main():
    wait_seconds = WAIT_MIN * 60
    async with async_playwright() as p:
        # ONE shared browser, headless — no visible windows, one OS process.
        browser = await p.chromium.launch(
            headless=HEADLESS,
            args=[
                '--no-sandbox',
                '--disable-setuid-sandbox',
                '--disable-dev-shm-usage',
                '--use-fake-ui-for-media-stream',
                '--use-fake-device-for-media-stream',
                '--disable-blink-features=AutomationControlled',
                '--disable-features=IsolateOrigins,site-per-process',
                '--window-size=1280,720',
            ],
        )
        try:
            tasks = [
                asyncio.create_task(
                    join_meeting(browser, f"[User-{i+1}]", MEETING_CODE, PASSCODE, wait_seconds)
                )
                for i in range(USERS)
            ]
            await asyncio.gather(*tasks)
        finally:
            await browser.close()
    print("\n✅ All users done!")


await main()
